<a href="https://colab.research.google.com/github/jonitodev/PembelajaranMesin/blob/main/JS04/JS04_00.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JS04 — Lab 0

Data: `Titanic-Dataset-selected.csv`. Unggah ke Files di Colab, lalu jalankan semua sel.

[Jobsheet Lab 0](https://polinema.gitbook.io/jti-modul-praktikum-pembelajaran-mesin-mah/js04-regresi/lab-0)

In [1]:
import pandas as pd
from pathlib import Path
from IPython.display import display
from sklearn.model_selection import train_test_split, KFold

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
DATA_DIR = Path('/content') if Path('/content').is_dir() else Path('.')

## Lab 0 — Pembagian data

In [2]:
titanic = pd.read_csv(DATA_DIR / 'Titanic-Dataset-selected.csv')
titanic.head()

,Survived,Pclass,Age,Sex,Cabin
0,0,3,-0.5925,1,115
1,1,1,0.6388,0,81
2,1,3,-0.2847,0,115
3,1,1,0.4079,0,55
4,0,3,0.4079,1,115


### Random split
Data dibagi menjadi train, validasi, dan test dengan rasio 80:10:10.

In [3]:
train_random, sisa_random = train_test_split(titanic, test_size=0.2, random_state=0)
val_random, test_random = train_test_split(sisa_random, test_size=0.5, random_state=0)

def ringkasan_split(bagian):
    return pd.DataFrame([
        {'Bagian': nama, 'Jumlah': len(df),
         'Label 0': (df.Survived == 0).sum(), 'Label 1': (df.Survived == 1).sum(),
         '% label 1': 100 * df.Survived.mean()}
        for nama, df in bagian.items()
    ])

display(ringkasan_split({'Asli': titanic, 'Train': train_random,
                        'Validasi': val_random, 'Test': test_random}))

,Bagian,Jumlah,Label 0,Label 1,% label 1
0,Asli,891,549,342,38.3838
1,Train,712,439,273,38.3427
2,Validasi,89,53,36,40.4494
3,Test,90,57,33,36.6667


### Stratified split
Stratifikasi menjaga proporsi label pada setiap bagian.

In [4]:
train_strat, sisa_strat = train_test_split(
    titanic, test_size=0.2, random_state=0, stratify=titanic.Survived)
val_strat, test_strat = train_test_split(
    sisa_strat, test_size=0.5, random_state=0, stratify=sisa_strat.Survived)
display(ringkasan_split({'Asli': titanic, 'Train': train_strat,
                        'Validasi': val_strat, 'Test': test_strat}))

,Bagian,Jumlah,Label 0,Label 1,% label 1
0,Asli,891,549,342,38.3838
1,Train,712,439,273,38.3427
2,Validasi,89,55,34,38.2022
3,Test,90,55,35,38.8889


### Cross validation 1
Seluruh data dibagi menjadi empat fold. KFold di sini tidak mengacak urutan baris.

In [5]:
folds = list(KFold(n_splits=4).split(titanic))
display(pd.DataFrame([
    {'Fold': i, 'Train': len(train_idx), 'Evaluasi': len(test_idx),
     'Indeks evaluasi': f'{test_idx[0]}–{test_idx[-1]}'}
    for i, (train_idx, test_idx) in enumerate(folds, 1)
]))
# Indeks lengkap tiap fold bisa dilihat pada variabel folds.

,Fold,Train,Evaluasi,Indeks evaluasi
0,1,668,223,0–222
1,2,668,223,223–445
2,3,668,223,446–668
3,4,669,222,669–890


### Cross validation 2
Sebanyak 20% data disimpan sebagai test. K-fold hanya dijalankan pada data train.

In [6]:
train_cv, test_cv = train_test_split(titanic, test_size=0.2, random_state=0)
folds_train = list(KFold(n_splits=4).split(train_cv))
display(pd.DataFrame([
    {'Fold': i, 'Train': len(train_idx), 'Validasi': len(val_idx), 'Test': len(test_cv)}
    for i, (train_idx, val_idx) in enumerate(folds_train, 1)
]))

,Fold,Train,Validasi,Test
0,1,534,178,179
1,2,534,178,179
2,3,534,178,179
3,4,534,178,179


Stratifikasi membuat proporsi label lebih mendekati data awal. Pada skema terakhir, test tetap terpisah selama validasi.